# GSB 5544 — Topic 9.1: K-Nearest Neighbors and Decision Trees for Regression  
*Three algorithms, one scikit-learn recipe. Watch each algorithm on the animation page, then reproduce what it did in pandas and scikit-learn. Fill each `____` blank as you work; the ✅ checks and the review questions ask for a sentence or a number.*

## The next 20 minutes

Last week the model was a line. This week two models make predictions in a different way: by **averaging
the nearest training rows** (k-nearest neighbors) and by **asking yes/no questions** (a decision tree).
The code barely changes; the idea behind the prediction does.

| | | tools |
|---|---|---|
| **1. One tiny data set** | eight houses: size → price, plus one new house | `pd.DataFrame`, `plot.scatter` |
| **2. Linear regression in motion** | the fitted line, the residuals, a prediction | `LinearRegression`, `.fit`, `.predict` |
| **3. KNN in motion** | the k nearest houses, their average, the staircase | `KNeighborsRegressor(n_neighbors=k)` |
| **4. Decision tree in motion** | score every split, keep the best, repeat, follow the questions | `DecisionTreeRegressor(max_depth=d)`, `plot_tree` |
| **5. Same recipe** | what changes between the three, what does not | `train_test_split`, `root_mean_squared_error` |
| **6. Definition: Pipeline** | preprocessing + model as one object | `Pipeline`, `ColumnTransformer`, `OneHotEncoder`, `StandardScaler` |
| **7. Review questions** | short checks on the animations and the workflow | |

🎬 **The animation page:** [https://gato365.github.io/gsb5544_instructor_learn_prep/week_9/algorithms.html](https://gato365.github.io/gsb5544_instructor_learn_prep/week_9/algorithms.html). It uses the same eight houses as this notebook, computes
everything in the browser, and highlights the line of scikit-learn code behind each step. Each algorithm has
its own address (`#linear-regression`, `#knn`, `#decision-tree`); the 🎬 links below open the page there.

**Rhythm for each section:** open the 🎬 link and watch → *predict* the number the notebook is about to print →
run the cell → compare with the page.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree

---
## 1. One tiny data set

Eight houses. **Size** is in hundreds of square feet, **price** in thousands of dollars. Small enough to
check every number by hand — which is the point. As in Topic 8.2, `X` is a DataFrame (double brackets) and
`y` is a Series.

In [ ]:
houses = pd.DataFrame({
    "size":  [10, 12, 15, 18, 20, 24, 28, 32],         # hundreds of square feet
    "price": [150, 180, 200, 260, 250, 320, 360, 410], # thousands of dollars
})
X = houses[[____]]
y = houses[____]
houses.plot.scatter(x="size", y="price");

And one **new house** of size 22.5 (2,250 square feet) whose price we want to predict. Every algorithm below
will be asked the same question about it. `predict` wants a DataFrame with the same column name as `X`.

In [ ]:
new = pd.DataFrame({"size": [____]})
new

---
## 2. Linear regression in motion

The model is a line, price ≈ intercept + slope × size. The animation shows the three things `fit` and
`predict` do: pick the line with the **smallest sum of squared residuals**, measure each **residual**
(actual − predicted, the vertical gap from a point to the line), and read a new size **off the line**.

🎬 *Watch it on the page:* [Linear regression: one line, chosen by least squares](https://gato365.github.io/gsb5544_instructor_learn_prep/week_9/algorithms.html#linear-regression) — press **Play**, or step through with → and ←.

In [ ]:
line = LinearRegression()
line.____(X, y)
print("intercept:", round(line.intercept_, 2), "  slope:", line.coef_.round(3))

The residuals, one per house, and their sum of squares. Compare the table with the page's table after the
residual steps.

In [ ]:
houses["predicted"] = line.predict(X)
houses["residual"]  = houses["price"] - houses[____]
print("sum of squared residuals:", round((houses["residual"] ** 2).sum(), 2))
houses.round(2)

In [ ]:
ax = houses.plot.scatter(x="size", y="price")
sizes = pd.DataFrame({"size": np.linspace(8, 34, 261)})
ax.plot(sizes["size"], line.predict(sizes), color="firebrick", label="fitted line")
ax.vlines(houses["size"], houses["predicted"], houses["price"], color="purple", label="residuals")
ax.legend();

In [ ]:
line.____(new)

✅ **Checkpoint 1.** (a) Read the slope in the data's units. (b) The residual for the size-20 house is about
−17.7: what does the minus sign say about that house? (c) Why can no other line have a smaller sum of squared
residuals than 715.1?

**Your answer:** *(write it here — replace this line)*

---
## 3. K-nearest neighbors in motion

KNN does not fit a shape at all. To predict a new house it **measures the distance to every training house,
keeps the k closest, and averages their prices**. `fit` only stores the training rows. With one feature the
distance is just the gap along the size axis, so no scaling is needed; section 5 says what changes with more
features.

🎬 *Watch it on the page:* [K-nearest neighbors: ask the k most similar houses](https://gato365.github.io/gsb5544_instructor_learn_prep/week_9/algorithms.html#knn) — press **Play**, or step through with → and ←.

First by hand, exactly as the animation does it: distance from every house to the new size, sort, keep three,
average.

In [ ]:
houses["distance"] = (houses["size"] - ____).abs()
nearest = houses.sort_values("distance").head(____)
print("prediction by hand:", round(nearest["price"].mean(), 2))
nearest[["size", "price", "distance"]]

Then with scikit-learn. Same recipe as the line: create, fit, predict. Only the first line changed.

In [ ]:
knn3 = KNeighborsRegressor(n_neighbors=____)
knn3.fit(X, y)
knn3.predict(new)

**The regression function.** Ask for a prediction at every size between 8 and 34 and the shape appears: a
**staircase**. The prediction changes only when the set of k nearest houses changes. Three values of k on one
plot, as in the last step of the animation (and as PA 9.1 asks you to sketch for the wine data).

In [ ]:
ax = houses.plot.scatter(x="size", y="price")
for k, color in zip([____, ____, ____], ["orange", "firebrick", "black"]):
    model = KNeighborsRegressor(n_neighbors=k).fit(X, y)
    ax.plot(sizes["size"], model.predict(sizes), color=color, label=f"k = {k}")
ax.legend();

✅ **Checkpoint 2.** (a) With k = 1, what is the prediction for size 22.5, and which house supplies it?
(b) Why is the k = 1 function a staircase rather than a line, and where exactly are its jumps? (c) Why is the
k = 8 function flat, and at what height?

**Your answer:** *(write it here — replace this line)*

---
## 4. Decision tree in motion

A tree predicts by asking **yes/no questions** about the features: "size ≤ 22?" Fitting means finding the
best question. For one numeric feature the candidate thresholds are the midpoints between neighboring sizes;
each candidate splits the houses into two groups, and its score is the **sum of squared errors (SSE)** of the
prices around the two group means. The smallest SSE wins. Then the same search runs again inside each group,
until `max_depth` levels of questions have been asked.

🎬 *Watch it on the page:* [Decision tree: split, then split again](https://gato365.github.io/gsb5544_instructor_learn_prep/week_9/algorithms.html#decision-tree) — press **Play**, or step through with → and ←.

The root split by hand: every candidate threshold and its SSE. (`sse` is the squared distance of each group's
prices from the group's own mean.)

In [ ]:
def sse(prices):
    return ((prices - prices.mean()) ** 2).sum()

sorted_sizes = sorted(houses["size"])
candidates = [(a + b) / 2 for a, b in zip(sorted_sizes[:-1], sorted_sizes[1:])]

rows = []
for t in candidates:
    left  = houses[houses["size"] ____ t]["price"]
    right = houses[houses["size"] > t]["price"]
    rows.append({"threshold": t, "left_mean": left.mean(), "right_mean": right.mean(),
                 "SSE": sse(left) + sse(right)})
pd.DataFrame(rows).round(1)

scikit-learn agrees. `max_depth=1` is a single question; `plot_tree` draws it with the SSE (shown as
`squared_error`, the mean per house), the number of houses and the mean price in each node.

In [ ]:
stump = DecisionTreeRegressor(max_depth=____).fit(X, y)
plot_tree(stump, feature_names=["size"], fontsize=10);

Two levels of questions. Follow the new house down: is 22.5 ≤ 22? No, go right. Is 22.5 ≤ 30? Yes, go left.
The leaf's mean price is the prediction.

In [ ]:
tree2 = DecisionTreeRegressor(max_depth=____).fit(X, y)
plt.figure(figsize=(9, 4))
plot_tree(tree2, feature_names=["size"], fontsize=10)
tree2.predict(new)

The regression function is again a staircase: **one flat value per leaf**. More depth means more leaves and
finer steps; at depth 3 every house but one gets its own leaf.

In [ ]:
ax = houses.plot.scatter(x="size", y="price")
for depth, color in zip([1, 2, 3], ["orange", "firebrick", "black"]):
    model = DecisionTreeRegressor(max_depth=____).fit(X, y)
    ax.plot(sizes["size"], model.predict(sizes), color=color, label=f"max_depth = {depth}")
ax.legend();

✅ **Checkpoint 3.** (a) Which threshold wins at the root, and which number decides it? (b) What does the
depth-2 tree predict for the new house, and which two questions did it answer on the way? (c) What happens to
the staircase as `max_depth` grows, and what word from Topic 8.2 describes a tree that is too deep?

**Your answer:** *(write it here — replace this line)*

---
## 5. Same recipe, different model

Everything you learned with `LinearRegression` carries over unchanged: **define `X` and `y` → split → create
the model → `.fit()` → `.predict()` → evaluate**. Read the three columns; only the shaded rows differ.

| step | linear regression | k-nearest neighbors | decision tree |
|---|---|---|---|
| import | `LinearRegression` | `KNeighborsRegressor` | `DecisionTreeRegressor` |
| define `X`, `y` | same | same | same |
| split | `train_test_split(...)` | same | same |
| **create** | `LinearRegression()` — no hyperparameters | `KNeighborsRegressor(n_neighbors=k)` | `DecisionTreeRegressor(max_depth=d)` |
| fit | `.fit(X_train, y_train)` | same | same |
| **what `.fit` learns** | intercept and slope | nothing: stores the rows | thresholds and leaf means |
| predict | `.predict(X_test)` | same | same |
| evaluate | `root_mean_squared_error(y_test, pred)` | same | same |
| **scale numeric features?** | not needed | **yes**, with more than one feature | not needed |
| categorical features | one-hot encode | one-hot encode | one-hot encode |
| **shape of the prediction** | line | staircase of local averages | staircase, one value per leaf |

**Why scaling matters only for KNN.** KNN measures distance, and a feature in square feet (hundreds to
thousands) would drown out a feature in bedrooms (1 to 5). `StandardScaler` puts every feature on the same
footing. A tree asks each question about one feature at a time, so the units of other features never
interfere; a linear model absorbs units into its coefficients. All three need categorical features turned
into numbers first (`OneHotEncoder`).

The Advertising data from Topic 8.2, three models, one loop. The pipeline around KNN is the subject of
section 6; for now, notice that the loop body is *identical* for all three.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

ads = pd.read_csv("https://www.statlearning.com/s/Advertising.csv", index_col=0)
X_ads = ads[["TV", "radio"]]
y_ads = ads["sales"]
X_train, X_test, y_train, y_test = train_test_split(X_ads, y_ads, test_size=0.25, random_state=5544)

models = {
    "linear regression":   LinearRegression(),
    "3-nearest neighbors": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=3)),
    "tree, max_depth=4":   DecisionTreeRegressor(max_depth=4, random_state=5544),
}
for name, model in models.items():
    model.____(X_train, y_train)
    rmse_train = root_mean_squared_error(y_train, model.predict(X_train))
    rmse_test  = root_mean_squared_error(y_test, model.____(X_test))
    print(f"{name:20s}  train RMSE = {rmse_train:.2f}   test RMSE = {rmse_test:.2f}")

✅ **Checkpoint 4.** (a) Which lines inside the loop would change if you added a fourth model? (b) Which model
predicts the test markets best here, and what is it about the data that a line cannot capture? (c) Why is the
tree's training RMSE so much better than its test RMSE compared with the line?

**Your answer:** *(write it here — replace this line)*

---
## 6. Definition: Pipeline

> 📦 **Definition — Pipeline.** *(Editable: reword this box as you like.)*
> A scikit-learn **Pipeline combines preprocessing steps and a model into one ordered workflow** that behaves
> like a single model: it has `.fit()` and `.predict()`, and those are the only two calls you make.
>
> - `pipe.fit(X_train, y_train)` runs the steps in order. Each preprocessing step **learns what it needs from
>   the training data** (a scaler learns the mean and standard deviation; a one-hot encoder learns the list of
>   categories), transforms the data and passes it on; the model at the end is fitted on the transformed data.
> - `pipe.predict(X_new)` applies **the same fitted preprocessing** to the new observations — the *training*
>   mean and standard deviation, the *training* categories; nothing is re-learned from the new rows — and then
>   asks the model.
>
> A pipeline is **not a prediction algorithm**. It makes no predictions of its own; it organizes the steps so
> that the right preprocessing always travels with the model.

Give the houses a categorical feature and the recipe needs preprocessing: one-hot encode `garage`, scale
`size`, then KNN. `ColumnTransformer` says which step applies to which column; `Pipeline` chains the
transformer and the model. This is exactly the structure PA 9.1 uses for the Ames neighborhoods.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

houses["garage"] = ["no", "no", "yes", "no", "yes", "yes", "yes", "yes"]
X2 = houses[["size", "garage"]]

preprocessing = ColumnTransformer([
    ("dummify", ____(), ["garage"]),
    ("scale",   ____(), ["size"]),
])
pipe = Pipeline([
    ("preprocessing", preprocessing),
    ("knn", KNeighborsRegressor(n_neighbors=3)),
])
pipe.____(X2, y)

new2 = pd.DataFrame({"size": [22.5], "garage": ["yes"]})
pipe.predict(new2)

What the new row looks like *after* the fitted preprocessing: two dummy columns and a scaled size. The scaled
value uses the training mean (19.875) and standard deviation (7.22): (22.5 − 19.875) / 7.22 ≈ 0.36.

In [ ]:
pd.DataFrame(pipe.named_steps["preprocessing"].transform(new2),
             columns=pipe.named_steps["preprocessing"].get_feature_names_out()).round(3)

✅ **Checkpoint 5.** (a) Is `pipe` a new kind of model? What does `pipe.predict` do with the new row's `garage`
value? (b) Why must the scaler use the *training* mean and standard deviation on new rows rather than
recomputing them? (c) The prediction changed from 276.67 to 310. Which three houses are the neighbors now?

**Your answer:** *(write it here — replace this line)*

---
## 7. Review questions

Short answers; a sentence or a number each.

**Q1.** In the linear-regression animation, what is a residual, and what exactly does "least squares"
minimize?

**Your answer:** *(write it here — replace this line)*

**Q2.** The KNN animation with k = 3 and a new house of size 22.5: which three houses are the neighbors, and
what is the prediction? What would the prediction be with k = 8, and why?

**Your answer:** *(write it here — replace this line)*

**Q3.** Why does a KNN regression function look like a staircase instead of a smooth curve?

**Your answer:** *(write it here — replace this line)*

**Q4.** In the tree animation, what number does the algorithm compute for each candidate split, and how does
it choose the split?

**Your answer:** *(write it here — replace this line)*

**Q5.** Write the one line that changes when you switch from linear regression to a 5-nearest-neighbor model
in the "same recipe" loop. Which lines stay the same?

**Your answer:** *(write it here — replace this line)*

**Q6.** You add `Bedroom AbvGr` (1–5) next to `Gr Liv Area` (300–5,000) as a second feature. Which of the three
models needs the features scaled, and why? What has to happen to `Neighborhood` before *any* of them can use it?

**Your answer:** *(write it here — replace this line)*

**Q7.** True or false, with a reason: "A Pipeline is a prediction algorithm, like KNN." What does a fitted
pipeline do with a new observation?

**Your answer:** *(write it here — replace this line)*

**Q8.** PA 9.1 asks what the 27-nearest-neighbor regression function looks like for the 27 training wines.
Answer it using what you saw with k = 8 on the houses.

**Your answer:** *(write it here — replace this line)*

---
## The lines to keep

| | |
|---|---|
| **Recipe, every model** | `model = Model(...)` → `model.fit(X_train, y_train)` → `model.predict(X_new)` → `root_mean_squared_error(y_test, model.predict(X_test))` |
| **Linear regression** | `.fit` learns intercept and slope by least squares; the prediction function is a line |
| **KNN** | `KNeighborsRegressor(n_neighbors=k)`: `.fit` stores the rows; `.predict` averages the k nearest; a staircase of local averages; scale features when there are several |
| **Decision tree** | `DecisionTreeRegressor(max_depth=d)`: `.fit` keeps the split with the smallest SSE and repeats; `.predict` follows the questions to a leaf; a staircase with one value per leaf; `plot_tree` to see it |
| **Hyperparameters** | `n_neighbors`, `max_depth`: chosen by you, compared on **test** error (Topic 8.2) |
| **Pipeline** | `Pipeline([("preprocessing", ColumnTransformer([...])), ("model", ...)])`: one object, same `.fit` / `.predict`; new rows get the *fitted* preprocessing |

Course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).